# Analysis - Runtime x Benchmark Performance Matrix

Post-run analysis of the TQ-VLM-Bench results. Loads all cell JSONs, builds a tidy DataFrame, and renders the full comparison grid with focused head-to-head cuts.

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))

from tq_bench.reporters import charts
from tq_bench.reporters.summary import render_markdown_summary

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

In [ ]:
RESULTS_DIR = Path('../../results').resolve()
runs_dir = RESULTS_DIR / 'runs'

rows = []
for p in sorted(runs_dir.glob('*.json')):
    with open(p) as f:
        r = json.load(f)
    rows.append({
        'runtime_id':          r.get('runtime_id'),
        'benchmark_id':        r.get('benchmark_id'),
        'score':               r.get('score'),
        'n_samples':           r.get('n_samples'),
        'n_failed':            r.get('n_failed', 0),
        'wall_time_seconds':   r.get('wall_time_seconds'),
        'status':              r.get('status', 'ok'),
    })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} cells')
df.head()

In [ ]:
# Full pivot: rows = runtime, cols = benchmark, values = score
pivot = df.pivot_table(
    index='runtime_id',
    columns='benchmark_id',
    values='score',
    aggfunc='mean',
)

# Keep matrix row order aligned with the runtimes table in CLAUDE.md
RUNTIME_ORDER = [
    'baseline', 'lcpp-kv-8', 'lcpp-kv-4', 'lcpp-kv-2',
    'tq-2', 'tq-2h', 'tq-3', 'tq-3h', 'tq-4',
    'tq-K4V2', 'tq-K4V3', 'tq-K3V2',
    'tqp-3', 'tqp-4', 'tqp-5',
]
pivot = pivot.reindex([r for r in RUNTIME_ORDER if r in pivot.index])
pivot.round(3)

In [ ]:
print(render_markdown_summary(df))

In [ ]:
# Heatmap of the full matrix
charts.heatmap(df)
plt.show()

## Key comparisons

- **lcpp-kv-4 vs tq-4**: same 4-bit budget, different method
- **lcpp-kv-2 vs tq-2**: same 2-bit budget
- **tq-3 vs tqp-3**: Algorithm 1 (MSE) vs Algorithm 2 (MSE+QJL)
- **tq-3 vs tq-K4V2**: same average 3-bit, symmetric vs asymmetric K/V

In [ ]:
def compare(runtimes):
    sub = pivot.loc[[r for r in runtimes if r in pivot.index]]
    return sub.round(3)

pairs = {
    '4-bit: lcpp vs TQ MSE':        ['baseline', 'lcpp-kv-4', 'tq-4'],
    '2-bit: lcpp vs TQ MSE':        ['baseline', 'lcpp-kv-2', 'tq-2'],
    'Alg1 vs Alg2 (3-bit)':         ['baseline', 'tq-3', 'tqp-3'],
    'Alg1 vs Alg2 (4-bit)':         ['baseline', 'tq-4', 'tqp-4'],
    'Symmetric vs asymmetric (avg 3)': ['baseline', 'tq-3', 'tq-K4V2'],
    'Symmetric vs asymmetric (avg 3.5)': ['baseline', 'tq-3h', 'tq-K4V3'],
}

for title, rts in pairs.items():
    print(f'### {title}')
    display(compare(rts))

## Degradation curves: score vs bitwidth (TQ MSE family)

In [ ]:
# Effective bits per runtime (TQ MSE family)
BITS = {
    'baseline': 16.0,
    'tq-2': 2.0, 'tq-2h': 2.5, 'tq-3': 3.0, 'tq-3h': 3.5, 'tq-4': 4.0,
    'lcpp-kv-2': 2.0, 'lcpp-kv-4': 4.0, 'lcpp-kv-8': 8.0,
    'tq-K4V2': 3.0, 'tq-K4V3': 3.5, 'tq-K3V2': 2.5,
    'tqp-3': 3.0, 'tqp-4': 4.0, 'tqp-5': 5.0,
}

charts.degradation_curve(df, bits_map=BITS)
plt.show()

## VLM vs Text sensitivity scatter

In [ ]:
charts.scatter_vlm_vs_text(df)
plt.show()

In [ ]:
# Save all charts to results/reports/
reports_dir = RESULTS_DIR / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
charts.generate_all_charts(df, reports_dir)
print(f'Charts written to {reports_dir}')